In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='darkgrid')

from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold, KFold, train_test_split
from sklearn.metrics import roc_auc_score

In [66]:
description = pd.read_excel('../data/description.xlsx')
X_train = pd.read_csv('../data/train_X.csv').fillna(0)
final_test = pd.read_csv('../data/test1_X.csv').fillna(0)
y_train = pd.read_csv('../data/train_y.csv')

In [67]:
groups = {
    "group_1": [
        "specialization_id", "contract_init_sum", 
         "project_id", "building_id", "contractor_id", 
        "contract_current_sum"
    ],
    "group_2": [
        "agg_all_contracts__g_contract__bit_da_guid__isMain__count__ALL_TIME", 
        "agg_all_contracts__g_contract__abs_change_price_last_ds__isMain__last__ALL_TIME", 
        "agg_all_contracts__g_contract__abs_change_price_last_ds__isMain__mean__ALL_TIME", 
        "agg_all_contracts__g_contract__rel_change_price_last_ds__isMain__last__ALL_TIME", 
        "agg_all_contracts__g_contract__rel_change_price_last_ds__isMain__mean__ALL_TIME"
    ],
    "group_3": [
        "agg_cec_requests__g_contract__request_id__all__count__1W",
        "agg_cec_requests__g_contract__request_id__all__count__2W",
        "agg_cec_requests__g_contract__request_id__all__count__3W",
        "agg_cec_requests__g_contract__request_id__all__count__4W",
        "agg_cec_requests__g_contract__request_id__all__count__5W",
        "agg_cec_requests__g_contract__request_id__all__count__6W",
        "agg_cec_requests__g_contract__request_id__all__count__7W",
        "agg_cec_requests__g_contract__request_id__all__count__8W",
        "agg_cec_requests__g_contract__request_id__all__count__12W",
        "agg_cec_requests__g_contract__request_id__all__count__ALL_TIME",
        "counteragent_sum_agg_cec_requests__g_contract__request_id__all__count__ALL_TIME",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__1W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__2W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__3W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__4W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__5W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__6W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__7W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__8W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__12W",
        "agg_cec_requests__g_contract__total_sum_accepted__all__sum__ALL_TIME",
        "counteragent_sum_agg_cec_requests__g_contract__total_sum_accepted__all__sum__ALL_TIME",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__1M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__2M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__3M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__4M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__5M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__6M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__7M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__8M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__12M",
        "agg_cec_requests__g_contract__time_btw_requests__all__mean__ALL_TIME",
        "agg_cec_requests__g_contract__created_dt__all__min__ALL_TIME",
        "agg_cec_requests__g_contract__created_dt__accepted__min__ALL_TIME"
    ],
    "group_4": [
        "agg_payments__g_contract__sum__all__countDistinct__1W",
        "agg_payments__g_contract__sum__all__countDistinct__2W",
        "agg_payments__g_contract__sum__all__countDistinct__4W",
        "agg_payments__g_contract__sum__all__countDistinct__8W",
        "agg_payments__g_contract__sum__all__countDistinct__12W",
        "agg_payments__g_contract__sum__all__countDistinct__ALL_TIME",
        "agg_payments__g_contract__sum__all__sum__1W",
        "agg_payments__g_contract__sum__all__sum__2W",
        "agg_payments__g_contract__sum__all__sum__4W",
        "agg_payments__g_contract__sum__all__sum__8W",
        "agg_payments__g_contract__sum__all__sum__12W",
        "agg_payments__g_contract__sum__all__sum__ALL_TIME",
        "counteragent_mean_agg_payments__g_contract__sum__all__sum__ALL_TIME",
        "counteragent_sum_agg_payments__g_contract__sum__all__sum__ALL_TIME",
        "specialization_sum_agg_payments__g_contract__sum__all__sum__ALL_TIME",
        "agg_payments__g_contract__date__advance__min__ALL_TIME"
    ],
    "group_5": [
        "agg_ks2__g_contract__id__all__count__1W",
        "agg_ks2__g_contract__id__all__count__2W",
        "agg_ks2__g_contract__id__all__count__4W",
        "agg_ks2__g_contract__id__all__count__8W",
        "agg_ks2__g_contract__id__all__count__12W",
        "agg_ks2__g_contract__id__all__count__ALL_TIME",
        "agg_ks2__g_contract__total_sum__all__sum__1W",
        "agg_ks2__g_contract__total_sum__all__sum__2W",
        "agg_ks2__g_contract__total_sum__all__sum__4W",
        "agg_ks2__g_contract__total_sum__all__sum__8W",
        "agg_ks2__g_contract__total_sum__all__sum__12W",
        "agg_ks2__g_contract__total_sum__all__sum__ALL_TIME",
        "counteragent_mean_agg_ks2__g_contract__total_sum__all__sum__ALL_TIME",
        "counteragent_sum_agg_ks2__g_contract__total_sum__all__sum__ALL_TIME",
        "specialization_sum_agg_ks2__g_contract__total_sum__all__sum__ALL_TIME"
    ],
    "group_6": [
        "agg_spass_applications__g_contract__appl_count_week__max__ALL_TIME",
        "agg_spass_applications__g_contract__appl_count_week__mean__1W",
        "agg_spass_applications__g_contract__appl_count_week__mean__2W",
        "agg_spass_applications__g_contract__appl_count_week__mean__3W",
        "agg_spass_applications__g_contract__appl_count_week__mean__4W",
        "agg_spass_applications__g_contract__appl_count_week__mean__5W",
        "agg_spass_applications__g_contract__appl_count_week__mean__6W",
        "agg_spass_applications__g_contract__appl_count_week__mean__8W",
        "agg_spass_applications__g_contract__appl_count_week__mean__12W",
        "agg_spass_applications__g_contract__appl_count_week__mean__26W",
        "agg_spass_applications__g_contract__appl_count_week__mean__ALL_TIME",
        "agg_spass_applications__g_specialization__appl_count_week__mean__ALL_TIME",
        "counteragent_mean_agg_spass_applications__g_contract__appl_count_week__mean__ALL_TIME"
    ],
    "group_7": [
        "agg_workers__g_contract__fact_workers__all__mean__1W",
        "agg_workers__g_contract__fact_workers__all__mean__2W",
        "agg_workers__g_contract__fact_workers__all__mean__3W",
        "agg_workers__g_contract__fact_workers__all__mean__4W",
        "agg_workers__g_contract__fact_workers__all__mean__5W",
        "agg_workers__g_contract__fact_workers__all__mean__6W",
        "agg_workers__g_contract__fact_workers__all__mean__8W",
        "agg_workers__g_contract__fact_workers__all__mean__12W",
        "agg_workers__g_contract__fact_workers__all__mean__26W",
        "agg_workers__g_contract__fact_workers__all__mean__ALL_TIME"
    ],
    "group_8": [
        'agg_materials__g_contract__order_id__countDistinct__1W',
        'agg_materials__g_contract__order_id__countDistinct__2W',
        'agg_materials__g_contract__order_id__countDistinct__4W',
        'agg_materials__g_contract__order_id__countDistinct__8W',
        'agg_materials__g_contract__order_id__countDistinct__12W',
        'agg_materials__g_contract__order_id__countDistinct__ALL_TIME',
        'agg_materials__g_contract__order_dt__min__ALL_TIME',
        'agg_materials__g_contract__material_type_id__countDistinct__ALL_TIME',
        'agg_materials__g_contract__material_id__countDistinct__ALL_TIME'
    ],
    "group_9": [
        'agg_Finance__g_contractor__Value__FixedAssets__last__ALL_TIME',
        'agg_Finance__g_contractor__Value__CurrentAssets__last__ALL_TIME',
        'agg_Finance__g_contractor__Value__Capital__last__ALL_TIME',
        'agg_Finance__g_contractor__Value__LongLiabilities__last__ALL_TIME',
        'agg_Finance__g_contractor__Value__ShortLiabilities__last__ALL_TIME',
        'agg_Finance__g_contractor__Value__Balance__last__ALL_TIME',
        'agg_Finance__g_contractor__Value__Revenue_y__last__ALL_TIME',
        'agg_Finance__g_contractor__Value__NetProfit_y__last__ALL_TIME',
        'agg_Finance__g_contractor__Value__CostPrice_y__last__ALL_TIME',
        'agg_FinanceAndTaxesFTS__g_contractor__Expenses__last__ALL_TIME',
        'agg_FinanceAndTaxesFTS__g_contractor__Income__last__ALL_TIME',
        'agg_FinanceAndTaxesFTS__g_contractor__TaxArrearsSum__last__ALL_TIME',
        'agg_FinanceAndTaxesFTS__g_contractor__TaxPenaltiesSum__last__ALL_TIME',
        'agg_FinanceAndTaxesFTS__g_contractor__TaxesSum__last__ALL_TIME'
    ]
}


In [68]:
X_train = X_train.drop(['contract_id' , 'contract_date', 'report_date'], axis=1)
final_test = final_test.drop(['contract_id' , 'contract_date', 'report_date'], axis=1)

In [69]:
# import pandas as pd
# from sklearn.cluster import KMeans

# # Допустим, X_train уже загружен и содержит все нужные данные

# # Функция для обучения KMeans и присвоения кластеров
# def apply_kmeans(df, features, num_clusters=5):
    
#     kmeans = KMeans(n_clusters=num_clusters, random_state=42)
#     clusters = kmeans.fit_predict(df[features])
    
#     return clusters

# # Применение кластеризации для каждой группы
# for group_name, features in groups.items():
#     X_train[group_name + "_cluster"] = apply_kmeans(X_train, features)

# kmeans_all = KMeans(n_clusters=8, random_state=42)
# X_train['cluster_all'] = kmeans_all.fit_predict(X_train)

In [70]:
contract_id = pd.read_csv('../data/train_X.csv')['contract_id'].values

In [71]:
y_train = y_train['default6']

In [72]:
def gini_score(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1
    return gini


In [73]:
model = CatBoostClassifier(iterations=1000, 
                           learning_rate=0.1, 
                           depth=5, 
                           eval_metric='AUC',
                           random_seed=42,
                           verbose=0)

gkf = GroupKFold(n_splits=5)
gini_scores = []

for train_index, test_index in gkf.split(X_train, y_train, groups=contract_id):
    X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
    
    model.fit(X_train_fold, y_train_fold, eval_set=(X_test_fold, y_test_fold), early_stopping_rounds=100, use_best_model=True)
    
    y_pred = model.predict_proba(X_test_fold)[:, 1]
    
    gini = gini_score(y_test_fold, y_pred)
    gini_scores.append(gini)
    print(f'Fold Gini Score: {gini}')

mean_gini_score = np.mean(gini_scores)
print(f'Mean Gini Score: {mean_gini_score}')


Fold Gini Score: 0.5254209016583338
Fold Gini Score: 0.5699899214245636
Fold Gini Score: 0.5609325760887762
Fold Gini Score: 0.5290059303250256
Fold Gini Score: 0.6056294524189263
Mean Gini Score: 0.5581957563831251


In [76]:
from xgboost import XGBClassifier

# Define a custom Gini scoring function
def gini_score(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

# Initialize the model with specific hyperparameters
model = XGBClassifier(
    eval_metric='auc',  # Evaluation metric
    random_state=42,    # Random seed
    use_label_encoder=False  # To suppress warnings about deprecation
)

# Set parameters using set_params
model.set_params(
    n_estimators=1000,  # Number of boosting rounds
    learning_rate=0.1,  # Step size
    max_depth=6         # Maximum depth of trees
)

# GroupKFold cross-validator
gkf = GroupKFold(n_splits=5)
gini_scores = []

# Perform cross-validation
for train_index, test_index in gkf.split(X_train, y_train, groups=contract_id):
    X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
    
    model.fit(X_train_fold, y_train_fold, eval_set=[(X_test_fold, y_test_fold)], early_stopping_rounds=100, verbose=0)
    
    y_pred = model.predict_proba(X_test_fold)[:, 1]
    
    gini = gini_score(y_test_fold, y_pred)
    gini_scores.append(gini)
    print(f'Fold Gini Score: {gini}')

mean_gini_score = np.mean(gini_scores)
print(f'Mean Gini Score: {mean_gini_score}')


/Users/miron/anaconda3/lib/python3.11/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold Gini Score: 0.5107829592219268


/Users/miron/anaconda3/lib/python3.11/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold Gini Score: 0.5862518511384822


/Users/miron/anaconda3/lib/python3.11/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold Gini Score: 0.5292139538209755


/Users/miron/anaconda3/lib/python3.11/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold Gini Score: 0.5089741637480332


/Users/miron/anaconda3/lib/python3.11/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold Gini Score: 0.591980595427964
Mean Gini Score: 0.5454407046714763


In [82]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

# Define a custom Gini scoring function
def gini_score(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

# Initialize the model with default parameters
model = lgb.LGBMClassifier(
    objective='binary',  # Binary classification
    eval_metric='auc',   # Evaluation metric
    random_state=42,     # Random seed
    verbosity=-1         # Suppress LightGBM output
)

# Set parameters using set_params
model.set_params(
    n_estimators=1000,   # Number of boosting rounds
    learning_rate=0.1,   # Step size
    max_depth=5,         # Maximum depth of trees
    num_leaves=31,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
    ]# Number of leaves in one tree
)

# GroupKFold cross-validator
gkf = GroupKFold(n_splits=5)
gini_scores = []

# Perform cross-validation
for train_index, test_index in gkf.split(X_train, y_train, groups=contract_id):
    X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
    
    # Fit the model with early stopping
    model.fit(
        X_train_fold, 
        y_train_fold, 
        eval_set=[(X_test_fold, y_test_fold)])
    
    # Predict probabilities and calculate Gini score
    y_pred = model.predict_proba(X_test_fold)[:, 1]
    gini = gini_score(y_test_fold, y_pred)
    gini_scores.append(gini)
    print(f'Fold Gini Score: {gini}')

# Calculate and print the mean Gini score
mean_gini_score = np.mean(gini_scores)
print(f'Mean Gini Score: {mean_gini_score}')


Fold Gini Score: 0.48347605588714737
Fold Gini Score: 0.545541183386602
Fold Gini Score: 0.49083416084160447
Fold Gini Score: 0.5049420080198976
Fold Gini Score: 0.5451127060074428
Mean Gini Score: 0.5139812228285389
